# AMOS External Test Dataset Evaluation

Evaluate AMOS CT organ images on OrganaMNIST-trained models.

**Dataset Info:**
- Source: AMOS_2022 CT scans (ID < 500)
- Total images: 4454 organ slices from 300 patients
- Format: 224×224 grayscale, HU windowed [-135, 215]

**Organ Mapping:**
- AMOS organs that match OrganaMNIST: spleen, kidneys (left/right), liver, pancreas
- OOD organs (not in OrganaMNIST): gall bladder, esophagus, stomach, aorta, postcava, adrenal glands, duodenum, bladder, prostate/uterus

## 1. Import Libraries and Load Data

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, 
    roc_auc_score, confusion_matrix,
    classification_report
)
import seaborn as sns
from tqdm import tqdm

# Set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load AMOS dataset
amos_path = '/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/AMOS_2022/amos_external_test_224.npz'
metadata_path = '/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/AMOS_2022/amos_external_test_224_metadata.json'

data = np.load(amos_path)
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

test_images = data['test_images']  # (N, 224, 224, 1)
test_labels = data['test_labels']  # (N, 15) - AMOS organ labels
test_sample_ids = data['test_sample_ids']  # (N,)

print(f"Loaded AMOS data:")
print(f"  Images: {test_images.shape}")
print(f"  Labels: {test_labels.shape}")
print(f"  Sample IDs: {test_sample_ids.shape}")
print(f"  Total samples: {len(test_images)}")

## 2. Create Label Mapping and Filter Dataset

In [ ]:
# AMOS organ IDs (from metadata)
amos_organs = {
    0: 'spleen',
    1: 'right kidney',
    2: 'left kidney',
    3: 'gall bladder',
    4: 'esophagus',
    5: 'liver',
    6: 'stomach',
    7: 'aorta',
    8: 'postcava',
    9: 'pancreas',
    10: 'right adrenal gland',
    11: 'left adrenal gland',
    12: 'duodenum',
    13: 'bladder',
    14: 'prostate/uterus'
}

# OrganaMNIST labels (from INFO)
organamnist_labels = {
    0: 'bladder',
    1: 'femur-left',
    2: 'femur-right',
    3: 'heart',
    4: 'kidney-left',
    5: 'kidney-right',
    6: 'liver',
    7: 'lung-left',
    8: 'lung-right',
    9: 'pancreas',
    10: 'spleen'
}

# Mapping: AMOS index → OrganaMNIST index (only for compatible organs)
amos_to_organamnist = {
    0: 10,  # spleen → spleen
    1: 5,   # right kidney → kidney-right
    2: 4,   # left kidney → kidney-left
    5: 6,   # liver → liver
    9: 9,   # pancreas → pancreas
    13: 0,  # bladder → bladder (URINARY bladder, not gall bladder!)
}

# OOD organs (not in OrganaMNIST, for UQ evaluation)
# Note: gall bladder (3) is DIFFERENT from bladder (13)!
ood_organs = {3, 4, 6, 7, 8, 10, 11, 12, 14}

print("✅ Organs mapped to OrganaMNIST:")
for amos_id, organ_id in amos_to_organamnist.items():
    print(f"   AMOS {amos_id:2d} ({amos_organs[amos_id]:20s}) → OrganaMNIST {organ_id:2d} ({organamnist_labels[organ_id]})")

print(f"\n⚠️  OOD organs (not in OrganaMNIST): {len(ood_organs)}")
for oid in sorted(ood_organs):
    print(f"   AMOS {oid:2d}: {amos_organs[oid]}")

In [ ]:
# Filter dataset: Keep only samples with mapped organs
mapped_indices = []
mapped_organamnist_labels = []

for idx in range(len(test_labels)):
    # Get AMOS organ ID (single organ per image)
    amos_organ_id = np.argmax(test_labels[idx])
    
    if amos_organ_id in amos_to_organamnist:
        mapped_indices.append(idx)
        mapped_organamnist_labels.append(amos_to_organamnist[amos_organ_id])

mapped_indices = np.array(mapped_indices)
mapped_organamnist_labels = np.array(mapped_organamnist_labels)

# Filter images
filtered_images = test_images[mapped_indices]
filtered_sample_ids = test_sample_ids[mapped_indices]

print(f"\n📊 Filtered dataset:")
print(f"  Original samples: {len(test_images)}")
print(f"  Mapped samples (ID-compatible): {len(filtered_images)}")
print(f"  Filtered out (OOD): {len(test_images) - len(filtered_images)}")

# Print class distribution
unique, counts = np.unique(mapped_organamnist_labels, return_counts=True)
print(f"\n🏷️  Class distribution in filtered dataset:")
for cls_id, count in zip(unique, counts):
    print(f"   Class {cls_id:2d} ({organamnist_labels[cls_id]:20s}): {count:4d} samples ({count/len(filtered_images)*100:5.1f}%)")

## 3. Create Dataset and DataLoader

In [ ]:
class AMOSDataset(Dataset):
    """AMOS external test dataset compatible with OrganaMNIST models."""
    
    def __init__(self, images, labels, transform=None):
        """
        Args:
            images: numpy array (N, 224, 224, 1) uint8
            labels: numpy array (N,) with OrganaMNIST class IDs
            transform: torchvision transforms
        """
        self.images = images
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Get image (H, W, 1) and convert to (H, W)
        img = self.images[idx].squeeze()  # (224, 224)
        label = self.labels[idx]
        
        # Convert to PIL Image for transforms (grayscale mode)
        from PIL import Image as PILImage
        img_pil = PILImage.fromarray(img, mode='L')
        
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        
        # Convert to 3-channel for ResNet (repeat grayscale)
        if img_tensor.shape[0] == 1:
            img_tensor = img_tensor.repeat(3, 1, 1)
        
        return img_tensor, torch.tensor(label, dtype=torch.long)

# Define transforms (match training preprocessing)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# Create dataset and dataloader
amos_dataset = AMOSDataset(filtered_images, mapped_organamnist_labels, transform=test_transform)
test_loader = DataLoader(amos_dataset, batch_size=64, shuffle=False, num_workers=4)

print(f"✅ Created DataLoader:")
print(f"   Total samples: {len(amos_dataset)}")
print(f"   Batch size: 64")
print(f"   Number of batches: {len(test_loader)}")

## 4. Load OrganaMNIST Models

In [ ]:
def load_organamnist_models(model_dir, num_models=5, use_augmented=False, device='cpu'):
    """Load trained OrganaMNIST ResNet18 models."""
    models = []
    num_classes = 11  # OrganaMNIST has 11 classes
    
    for i in range(num_models):
        # Initialize ResNet18
        model = resnet18(weights=ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
        # Load weights
        suffix = '_augmented' if use_augmented else ''
        model_path = os.path.join(model_dir, f'resnet18_organamnist_224_{i}{suffix}.pt')
        
        if not os.path.exists(model_path):
            print(f"⚠️  Warning: Model not found: {model_path}")
            continue
        
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        model.to(device)
        model.eval()
        models.append(model)
        print(f"✅ Loaded model {i}: {os.path.basename(model_path)}")
    
    return models

# Load models
model_dir = '/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/models/224*224'
models = load_organamnist_models(model_dir, num_models=5, use_augmented=False, device=device)
models_augmented = load_organamnist_models(model_dir, num_models=5, use_augmented=True, device=device)

print(f"\n📊 Loaded {len(models)} OrganaMNIST models")
print(f"\n📊 Loaded {len(models_augmented)} OrganaMNIST models w data augmentation")

## 5. Evaluate on AMOS Test Data

In [ ]:
def evaluate_ensemble(models, dataloader, device):
    """Evaluate ensemble of models on AMOS test data."""
    all_preds = []
    all_probs = []
    all_labels = []
    all_individual_probs = [[] for _ in range(len(models))]
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)
            
            # Get predictions from each model
            batch_logits = []
            for i, model in enumerate(models):
                logits = model(images)
                probs = F.softmax(logits, dim=1)
                batch_logits.append(logits)
                all_individual_probs[i].append(probs.cpu().numpy())
            
            # Ensemble prediction (average probabilities)
            ensemble_probs = torch.stack([F.softmax(logits, dim=1) for logits in batch_logits]).mean(dim=0)
            ensemble_preds = ensemble_probs.argmax(dim=1)
            
            all_probs.append(ensemble_probs.cpu().numpy())
            all_preds.append(ensemble_preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    # Concatenate all batches
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_individual_probs = [np.concatenate(probs, axis=0) for probs in all_individual_probs]
    
    return all_preds, all_probs, all_labels, all_individual_probs

# Run evaluation
print("🔍 Evaluating ensemble on AMOS external test data...")
predictions, probabilities, true_labels, individual_probs = evaluate_ensemble(models, test_loader, device)
predictions_aug, probabilities_aug, true_labels_aug, individual_probs_aug = evaluate_ensemble(models_augmented, test_loader, device)

print(f"\n✅ Evaluation complete!")
print(f"   Predictions shape: {predictions.shape}")
print(f"   Probabilities shape: {probabilities.shape}")
print(f"   True labels shape: {true_labels.shape}")

print(f"\n✅ Evaluation complete for augmented models!")
print(f"   Predictions shape: {predictions_aug.shape}")
print(f"   Probabilities shape: {probabilities_aug.shape}")
print(f"   True labels shape: {true_labels_aug.shape}")

## 6. Compute Metrics

In [ ]:
# Compute metrics
acc = accuracy_score(true_labels, predictions)
bal_acc = balanced_accuracy_score(true_labels, predictions)

# For AUC: compute using label binarization for only the classes present in AMOS
from sklearn.preprocessing import label_binarize
unique_classes = np.unique(true_labels)
# Binarize labels for OvR AUC computation
y_true_bin = label_binarize(true_labels, classes=unique_classes)
# Extract probabilities for only the present classes
y_score_filtered = probabilities[:, unique_classes]
# Compute macro AUC
auc = roc_auc_score(y_true_bin, y_score_filtered, average='macro')

print("=" * 60)
print("AMOS EXTERNAL TEST SET EVALUATION")
print("=" * 60)
print(f"\n📊 Overall Performance:")
print(f"   Accuracy:          {acc:.4f}")
print(f"   Balanced Accuracy: {bal_acc:.4f}")
print(f"   Macro AUC (OvR):   {auc:.4f} (computed for {len(unique_classes)} present classes)")

# Per-class metrics (only for classes present in AMOS data)
print(f"\n🏷️  Per-Class Performance:")
print(f"Classes present in AMOS: {sorted(unique_classes)}")
print(classification_report(
    true_labels, predictions,
    target_names=[organamnist_labels[i] for i in unique_classes],
    labels=unique_classes,
    digits=3
))

## 7. Visualize Confusion Matrix

In [ ]:
# Confusion matrix (show ALL 11 OrganaMNIST classes to see predictions)
# Rows: only AMOS classes (6), Columns: all OrganaMNIST classes (11)
all_classes = list(range(11))  # All OrganaMNIST classes
amos_classes = np.unique(true_labels)  # Only AMOS classes

# Full confusion matrix: AMOS true labels (rows) vs all OrganaMNIST predictions (cols)
cm_full = confusion_matrix(true_labels, predictions, labels=all_classes)
# Filter to only rows with AMOS classes
cm_filtered = cm_full[amos_classes, :]

# Normalize by row (true labels) - shows distribution of predictions for each true class
cm_norm = cm_filtered.astype('float') / cm_filtered.sum(axis=1, keepdims=True)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Absolute counts
sns.heatmap(cm_filtered, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=[organamnist_labels[i] for i in all_classes],
            yticklabels=[organamnist_labels[i] for i in amos_classes])
ax1.set_xlabel('Predicted (All 11 OrganaMNIST Classes)', fontsize=12)
ax1.set_ylabel('True (AMOS Classes)', fontsize=12)
ax1.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
            xticklabels=[organamnist_labels[i] for i in all_classes],
            yticklabels=[organamnist_labels[i] for i in amos_classes])
ax2.set_xlabel('Predicted (All 11 OrganaMNIST Classes)', fontsize=12)
ax2.set_ylabel('True (AMOS Classes)', fontsize=12)
ax2.set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('amos_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to: amos_confusion_matrix.png")
print(f"\n📊 Predictions summary:")
print(f"   Classes present in AMOS (rows): {sorted(amos_classes)}")
for cls_id in all_classes:
    pred_count = (predictions == cls_id).sum()
    if pred_count > 0:
        pct = pred_count / len(predictions) * 100
        in_amos = '✓' if cls_id in amos_classes else '✗ (NOT in AMOS!)'
        print(f"   Class {cls_id:2d} ({organamnist_labels[cls_id]:15s}): {pred_count:4d} predictions ({pct:5.2f}%) {in_amos}")

In [ ]:
# Compute metrics
acc_aug = accuracy_score(true_labels, predictions_aug)
bal_acc_aug = balanced_accuracy_score(true_labels, predictions_aug)

# For AUC: compute using label binarization for only the classes present in AMOS
y_score_filtered_aug = probabilities_aug[:, unique_classes]
# Compute macro AUC
auc_aug = roc_auc_score(y_true_bin, y_score_filtered_aug, average='macro')

print("=" * 60)
print("AMOS EXTERNAL TEST SET EVALUATION DATA AUGMENTATION")
print("=" * 60)
print(f"\n📊 Overall Performance:")
print(f"   Accuracy:          {acc_aug:.4f}")
print(f"   Balanced Accuracy: {bal_acc_aug:.4f}")
print(f"   Macro AUC (OvR):   {auc_aug:.4f} (computed for {len(unique_classes)} present classes)")

# Per-class metrics (only for classes present in AMOS data)
print(f"\n🏷️  Per-Class Performance:")
print(f"Classes present in AMOS: {sorted(unique_classes)}")
print(classification_report(
    true_labels, predictions_aug,
    target_names=[organamnist_labels[i] for i in unique_classes],
    labels=unique_classes,
    digits=3
))

In [ ]:
# Confusion matrix (show ALL 11 OrganaMNIST classes to see predictions)
# Rows: only AMOS classes (6), Columns: all OrganaMNIST classes (11)

# Full confusion matrix: AMOS true labels (rows) vs all OrganaMNIST predictions (cols)
cm_full_aug = confusion_matrix(true_labels, predictions_aug, labels=all_classes)
# Filter to only rows with AMOS classes
cm_filtered_aug = cm_full_aug[amos_classes, :]

# Normalize by row (true labels) - shows distribution of predictions for each true class
cm_norm_aug = cm_filtered_aug.astype('float') / cm_filtered_aug.sum(axis=1, keepdims=True)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Absolute counts
sns.heatmap(cm_filtered_aug, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=[organamnist_labels[i] for i in all_classes],
            yticklabels=[organamnist_labels[i] for i in amos_classes])
ax1.set_xlabel('Predicted (All 11 OrganaMNIST Classes)', fontsize=12)
ax1.set_ylabel('True (AMOS Classes)', fontsize=12)
ax1.set_title('Confusion Matrix - Augmented (Counts)', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Normalized
sns.heatmap(cm_norm_aug, annot=True, fmt='.2f', cmap='Blues', ax=ax2,
            xticklabels=[organamnist_labels[i] for i in all_classes],
            yticklabels=[organamnist_labels[i] for i in amos_classes])
ax2.set_xlabel('Predicted (All 11 OrganaMNIST Classes)', fontsize=12)
ax2.set_ylabel('True (AMOS Classes)', fontsize=12)
ax2.set_title('Confusion Matrix - Augmented (Normalized)', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('amos_confusion_augmented_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to: amos_confusion_augmented_matrix.png")
print(f"\n📊 Predictions summary (Augmented):")
print(f"   Classes present in AMOS (rows): {sorted(amos_classes)}")
for cls_id in all_classes:
    pred_count = (predictions_aug == cls_id).sum()
    if pred_count > 0:
        pct = pred_count / len(predictions_aug) * 100
        in_amos = '✓' if cls_id in amos_classes else '✗ (NOT in AMOS!)'
        print(f"   Class {cls_id:2d} ({organamnist_labels[cls_id]:15s}): {pred_count:4d} predictions ({pct:5.2f}%) {in_amos}")